In [1]:
# QA pipeline for processing summaries and segments
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from pathlib import Path
import json
import os

# Initialize the QA model
qa_tokenizer = AutoTokenizer.from_pretrained("valhalla/longformer-base-4096-finetuned-squadv1")
qa_model = AutoModelForQuestionAnswering.from_pretrained("valhalla/longformer-base-4096-finetuned-squadv1")

Some weights of the model checkpoint at valhalla/longformer-base-4096-finetuned-squadv1 were not used when initializing LongformerForQuestionAnswering: ['longformer.pooler.dense.bias', 'longformer.pooler.dense.weight']
- This IS expected if you are initializing LongformerForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing LongformerForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# Define separate question sets
summary_questions = [
    "What is the main topic of this story?",
    "Who are the key people mentioned?",
    "What are the major events described?",
    "What is the most important insight?"
]

segment_questions = [
    "What is being discussed in this audio segment?",
    "Who are the speakers or participants involved?",
    "What emotions or tones are expressed in this part?",
    "Are there any important decisions or agreements made?",
    "What is the main purpose or intent behind the conversation?",
    "Is there any conflict or disagreement in this segment?",
    "What insights can be inferred from the speaker's tone or language?"
]


# Process summary file
summary_file = Path("summaries_output.txt")
summary_qa_results = {}

if summary_file.exists():
    print("Processing summaries_output.txt...")
    with open(summary_file, "r", encoding="utf-8", errors="ignore") as f:
        summary_text = f.read()
    
    summary_qa_results["file"] = str(summary_file)
    summary_qa_results["qa_pairs"] = []
    
    for question in summary_questions:
        answer = answer_question(question, summary_text)
        summary_qa_results["qa_pairs"].append({
            "question": question,
            "answer": answer["answer"],
            "confidence": answer["confidence"]
        })
        print(f"Q: {question}\nA: {answer['answer']} (Confidence: {answer['confidence']})\n")

# Process segment files
segments_dir = Path("segments")
segment_qa_results = []

if segments_dir.exists():
    print("\nProcessing segment files...")
    segment_files = sorted([f for f in segments_dir.glob("segment_*_response.txt")])
    
    for segment_file in segment_files:
        if segment_file.exists():
            with open(segment_file, "r", encoding="utf-8", errors="ignore") as f:
                segment_text = f.read()
            
            segment_result = {
                "file": str(segment_file),
                "qa_pairs": []
            }
            
            print(f"\nProcessing {segment_file.name}:")
            for question in segment_questions:
                answer = answer_question(question, segment_text)
                segment_result["qa_pairs"].append({
                    "question": question,
                    "answer": answer["answer"],
                    "confidence": answer["confidence"]
                })
                print(f"Q: {question}\nA: {answer['answer']} (Confidence: {answer['confidence']})")
            
            segment_qa_results.append(segment_result)

# Save results to JSON files
with open("summary_qa_results.json", "w", encoding="utf-8") as f:
    json.dump(summary_qa_results, f, indent=2)

with open("segment_qa_results.json", "w", encoding="utf-8") as f:
    json.dump(segment_qa_results, f, indent=2)

print("\nQA processing complete. Results saved to summary_qa_results.json and segment_qa_results.json")


Processing summaries_output.txt...


NameError: name 'answer_question' is not defined

: 